# 3. Final Pipeline

This notebook is the single entry point for the project. It

- runs feature engineering (notebook 1) then the four per-instrument metamodels (2a to 2d),
- assembles the one required deliverable, `data/deliverables/predictions.csv`, in the `date,instrument,prediction` format from the brief, covering H1 2022 for the Energy asset class (cl1s, ho1s, rb1s, ng1s),
- prints the winning-model comparison and the prediction graphs.

Set `RUN_NOTEBOOKS = True` for a clean end-to-end run, or `False` to rebuild the deliverable from the prediction CSVs already on disk.

## Results summary

The figures in the table below are those we recorded running the full pipeline ourselves on a MacBook Air (M1). We fix every seed and pin BLAS and TensorFlow to a single thread, so a clean top-to-bottom run on the same machine and library versions reproduces them. A rerun on different hardware, or on the hidden H2 2022 window, can still shift the numbers. Because the top models sit within one cross-validation standard deviation of each other, even the winning model per instrument can change. The live figures for any given run are printed in Section 11.1 of each model notebook. The regenerated predictions.csv and the Section 4 plots are that run's actual output, so they need not match this fixed reference table.

| Instrument | Winning model | CV AUC | Full-test AUC | Leak-free OOS AUC | Verdict on H1 2022 |
|---|---|---|---|---|---|
| RBOB gasoline (rb1s) | MLP | 0.65 | 0.60 | 0.43 | class-inversion on the OOS slice, does not beat the blind primary |
| Heating oil (ho1s) | Logistic Regression (override) | 0.54 | 0.60 | n=1 | structurally unevaluable, only one OOS trade |
| Natural gas (ng1s) | Logistic Regression | 0.89 | 0.63 | 0.62 | the only clean win, lifts precision from 0.26 to 0.30 |
| WTI crude (cl1s) | Logistic Regression | 0.67 | 0.66 | 0.33 | class-inversion on the OOS slice, does not beat the blind primary |

Two instruments (RB1S and CL1S) have a flipped sign out of sample, and one instrument (HO1S) has too few trades to be evaluated. This is reported through a leak-free audit rather than the aggregate test. The qualitative pattern, one clean win, two out-of-sample inversions and one unevaluable instrument, is what we expect to persist across reruns even if the exact AUCs move.

## 1. Run the full pipeline

In the configuration cell below, `PYTHON_EXE` is the one machine-specific setting. Point it at the absolute path of the Python interpreter that has this project's dependencies installed (numpy, tensorflow, xgboost, hmmlearn, and so on). nbconvert registers a Jupyter kernel bound to exactly this interpreter and runs every child notebook under it, so getting this right is what avoids spurious `ModuleNotFoundError`s.

In [ ]:
import sys, os, time, json, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# When run with Jupyter, the working directory is the notebooks/ folder.
NB_DIR   = Path.cwd()
REPO     = NB_DIR.parent
DELIV    = REPO / 'data' / 'deliverables'
EXEC_DIR = NB_DIR / '_executed'          # executed copies land here, source notebooks stay clean

# Run order: feature engineering first (produces features.parquet), then the four meta-models.
PIPELINE = [
    '1_feature_engineering.ipynb',
    '2a_rbob_gasoline_model_training.ipynb',
    '2b_heating_oil_model_training.ipynb',
    '2c_natural_gas_model_training.ipynb',
    '2d_wti_crude_oil_model_training.ipynb',
]

# Energy asset class = the instruments we cover end-to-end.
INSTRUMENTS = {
    'cl1s': 'WTI Crude Oil',
    'ho1s': 'Heating Oil',
    'rb1s': 'RBOB Gasoline',
    'ng1s': 'Natural Gas',
}

# Set False to skip the (slow) re-execution and just rebuild the deliverable + plots from the
# prediction CSVs already on disk. Set True for a clean end-to-end reproducibility run.
RUN_NOTEBOOKS  = True
CELL_TIMEOUT_S = 3600   # per-cell timeout, the neural-net training notebooks are the slow ones

# Interpreter used to run the per-notebook subprocesses below. By default this is the same
# Python that is running this notebook. Point it at another environment if you want the
# pipeline to run under a specific venv or conda env, for example:
#     PYTHON_EXE = '/opt/homebrew/bin/python3.10'
#     PYTHON_EXE = r'C:\\Users\\me\\miniconda3\\envs\\sts\\python.exe'
PYTHON_EXE = "/opt/homebrew/bin/python3.10"

# IMPORTANT: nbconvert executes each child notebook through a *registered Jupyter kernel*, not
# through PYTHON_EXE directly. The notebooks' saved kernel ('python3') can resolve to a
# different environment than PYTHON_EXE (e.g. another venv, or one without hmmlearn), which is
# what produces a spurious 'ModuleNotFoundError: No module named hmmlearn' even though
# PYTHON_EXE has it installed. The execution cell below registers a Jupyter kernel bound to
# exactly PYTHON_EXE and forces nbconvert to use it, so the children always run under this
# interpreter regardless of their saved kernelspec or the current PATH.
KERNEL_NAME = 'stt-pipeline'

# Path made visible to the child notebooks so 'from src.part5 import ...' always resolves,
# whichever interpreter runs them. The notebooks also self-discover the repo root, so this
# is belt and braces.
RUN_PYTHONPATH = str(REPO) + os.pathsep + os.environ.get('PYTHONPATH', '')

print('repo        :', REPO)
print('deliverables:', DELIV)
print('interpreter :', PYTHON_EXE)
print('kernel      :', KERNEL_NAME)
print('run order   :', PIPELINE)

We execute each notebook with `jupyter nbconvert --execute`, each in its own directory so the `../data/...` paths resolve. Executed copies go to `notebooks/_executed/` to keep the sources clean. If a notebook fails we record it and carry on, then fall back to any prediction CSVs already on disk.

In [ ]:
status = []
if RUN_NOTEBOOKS:
    EXEC_DIR.mkdir(exist_ok=True)
    run_env = dict(os.environ, PYTHONPATH=RUN_PYTHONPATH)

    # Register (idempotently) a user-level Jupyter kernel bound to PYTHON_EXE, then make
    # nbconvert run every child notebook under it. Without this, nbconvert falls back to each
    # notebook's saved kernel ('python3'), which may resolve to a different environment and
    # fail with e.g. 'No module named hmmlearn'. ipykernel writes the absolute PYTHON_EXE path
    # into the kernelspec, so this pins execution to exactly the interpreter selected above.
    reg = subprocess.run(
        [PYTHON_EXE, '-m', 'ipykernel', 'install', '--user',
         '--name', KERNEL_NAME, '--display-name', f'STT pipeline ({PYTHON_EXE})'],
        capture_output=True, text=True,
    )
    if reg.returncode != 0:
        raise RuntimeError(f'could not register Jupyter kernel for {PYTHON_EXE}:\n{reg.stderr}')
    print(f'using kernel {KERNEL_NAME!r} -> {PYTHON_EXE}\n')

    for nb in PIPELINE:
        t0 = time.time()
        cmd = [
            PYTHON_EXE, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute',
            str(NB_DIR / nb), '--output-dir', str(EXEC_DIR),
            f'--ExecutePreprocessor.kernel_name={KERNEL_NAME}',
            f'--ExecutePreprocessor.timeout={CELL_TIMEOUT_S}',
        ]
        proc = subprocess.run(cmd, capture_output=True, text=True, env=run_env)
        ok = proc.returncode == 0
        mins = round((time.time() - t0) / 60, 1)
        status.append({'notebook': nb, 'status': 'ok' if ok else 'FAILED', 'minutes': mins})
        print(f"{'ok  ' if ok else 'FAIL'}  {nb:<42}  {mins:>5} min")
        if not ok:
            print('  --- last lines of stderr ---')
            print('\n'.join(proc.stderr.strip().splitlines()[-15:]))
else:
    print('RUN_NOTEBOOKS = False  ->  skipping execution, using existing CSVs in', DELIV)

run_log = pd.DataFrame(status)
run_log

## 2. Assemble the combined deliverable CSV

We read each per-instrument `predictions_<ticker>.csv`, concatenate them, validate the format required by the brief (`date,instrument,prediction`, one row per pair, probability in [0, 1], dates within 2022), sort by (date, instrument), and write the single combined `predictions.csv`. On the released data this is H1 2022. On the hidden rerun the export window extends automatically, so the same code emits the full 2022 window.

In [ ]:
frames = []
for ins in INSTRUMENTS:
    f = DELIV / f'predictions_{ins}.csv'
    if not f.exists():
        print('WARNING: missing', f.name, '- skipping')
        continue
    frames.append(pd.read_csv(f, parse_dates=['date']))

combined = pd.concat(frames, ignore_index=True)

# --- validate the deliverable format ---
assert list(combined.columns) == ['date', 'instrument', 'prediction'], combined.columns.tolist()
assert combined['prediction'].between(0, 1).all(), 'predictions must lie in [0, 1]'
assert not combined.duplicated(['date', 'instrument']).any(), 'one row per (date, instrument)'
assert (combined['date'].dt.year == 2022).all(), 'predictions must fall within 2022'

combined = combined.sort_values(['date', 'instrument']).reset_index(drop=True)
combined['date'] = combined['date'].dt.strftime('%Y-%m-%d')

out_path = DELIV / 'predictions.csv'
combined.to_csv(out_path, index=False)
print(f'Wrote {len(combined)} rows to {out_path}')
print('Instruments :', sorted(combined["instrument"].unique()))
print('Date range  :', combined["date"].min(), '->', combined["date"].max())
combined.head(8)

## 3. Winning model per instrument

The table and chart below reproduce the per-instrument comparison from the results summary at the top, using the figures we recorded running the pipeline ourselves on a MacBook Air (M1): the walk-forward CV AUC (the selection metric), the full H1 2022 holdout test AUC, and the leak-free Section 11.1 out-of-sample AUC. They are kept as fixed reference numbers rather than read back from the model notebooks, because a rerun on different hardware can move the numbers and even change which model wins. The live figures for any given run are printed by Section 11.1 of each model notebook (Aug to Dec 2021 selection slice, Jan to Jun 2022 OOS slice, frozen threshold, blind baseline, degeneracy check), which is also where rb1s and cl1s show the OOS class-inversion.

In [ ]:
# Per-instrument winners. These are the figures we recorded running the full
# pipeline ourselves on a MacBook Air (M1): cv_auc is the winning model's mean
# walk-forward CV AUC (Section 11), test_auc its full H1 2022 holdout AUC
# (Section 9/10), oos_auc the leak-free Jan-Jun 2022 reading from the Section 11.1
# audit (None when the OOS slice is too small to evaluate, as for heating oil).
# rb1s and cl1s show OOS class-inversion (AUC below 0.45).
#
# They are kept here as fixed reference values rather than read back from the model
# notebooks, because a rerun on different hardware can shift the numbers and even
# change which model wins. The live figures for any given run are printed in
# Section 11.1 of each model notebook.
RECORDED = {
    'rb1s': {'winning_model': 'MLP',                 'cv_auc': 0.6530, 'test_auc': 0.6016, 'oos_auc': 0.430},
    'ho1s': {'winning_model': 'Logistic Regression', 'cv_auc': 0.5382, 'test_auc': 0.5972, 'oos_auc': None},
    'ng1s': {'winning_model': 'Logistic Regression', 'cv_auc': 0.8889, 'test_auc': 0.6270, 'oos_auc': 0.621},
    'cl1s': {'winning_model': 'Logistic Regression', 'cv_auc': 0.6654, 'test_auc': 0.6618, 'oos_auc': 0.334},
}

rows = []
for ins, name in INSTRUMENTS.items():
    m = RECORDED[ins]
    rows.append({
        'instrument':    ins,
        'name':          name,
        'winning_model': m['winning_model'],
        'cv_auc':        m['cv_auc'],
        'test_auc':      m['test_auc'],
        'oos_auc':       m['oos_auc'] if m['oos_auc'] is not None else np.nan,
    })

winners = pd.DataFrame(rows)
print(winners.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(winners))
ax.barh(y + 0.2, winners['cv_auc'],   height=0.4, color='steelblue', alpha=0.85, label='Walk-forward CV AUC')
ax.barh(y - 0.2, winners['test_auc'], height=0.4, color='indianred', alpha=0.85, label='Full-test AUC (H1 2022 holdout)')
ax.axvline(0.5, color='black', lw=0.8, linestyle=':', label='random (0.50)')
ax.set_yticks(y)
ax.set_yticklabels(winners['name'] + '\n(' + winners['winning_model'] + ')')
ax.set_xlabel('AUC')
ax.set_xlim(0, 1)
ax.set_title('Winning model per instrument: CV vs full-test AUC')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 3.1 Per-instrument interpretation

The four metamodels share one pipeline but behave very differently once we look past the headline AUC. The following analysis extracts the model interpretation and the honest value-added verdict from Section 12.3 and Section 11.1 of each per-instrument notebook.

**WTI crude oil (cl1s), logistic regression:** 
This is the instrument with the most genuine structure. Four feature clusters earn a positive permutation drop on both the linear model and the MLP, which is the strongest sign that the signal is real and not an artefact of one estimator: cross-sectional and seasonal coherence (the largest driver), regime age, the latent HMM regime block, and trend strength. That is exactly the profile we would want from a metamodel sitting on a trend-following primary signal. Even so, it does not add value on the deliverable window. The full-test AUC of 0.6618 is the highest in the book, but the leak-free Section 11.1 audit shows that this number was carried entirely by the August to December 2021 half. On the January to June 2022 out-of-sample slice, the AUC collapses to 0.334, the audit raises a class-inversion flag, and at the frozen threshold, the filter sits below the blind primary on precision, recall, and F1. Honest verdict: a well-behaved model in-sample whose edge does not survive the early-2022 regime break.

**Natural gas (ng1s), logistic regression:** 
The model leans on the prevailing market regime (the `regime_4` cluster is the top out-of-sample driver for both model families) and on broad energy-spread conditions, while implied volatility carries a large in-sample coefficient that does not survive out of sample. That last point is unsurprising because the feature is built from OVX, a crude-oil proxy rather than a gas one. This is the one clean win. The out-of-sample AUC of 0.621 is in line with the full-test 0.627. The degeneracy audit raises no flags, and at the frozen threshold, the filter buys precision over the blind primary. It lifts precision from about 0.26 to about 0.30 at the cost of roughly 40 per cent of recall, which is the right shape for a precision-buying filter rather than a generic accuracy gain. Honest verdict: a modest but audited and genuine edge, and the cleanest single-instrument result in the book.

**RBOB gasoline (rb1s), MLP:** 
Interpretation rests on essentially one pillar: the latent HMM regime state. This pillar is corroborated in three ways: the MLP permutation importance, the cross-family logistic-regression importance, and the SHAP attribution all agree on it. That fits gasoline as a seasonal, demand-driven refined product, and everything else is small and noisy. On value, the model does not beat the blind primary in H1 2022. On the selection slice, the threshold looked genuinely selective (precision of about 0.84 while taking under half the signals). However, when frozen onto the out-of-sample window, the probability distribution shifts up, the same threshold takes almost every signal, precision falls below the base rate, and the out-of-sample AUC is 0.430 with a class-inversion flag. Honest verdict: real in-sample regime conditioning whose selectivity does not transfer through the regime break.

**Heating oil (ho1s), logistic regression by deliberate override:** 
The feature picture is the weakest and least stable in the book. Petroleum product and crack spreads are the strongest out-of-sample driver for the linear model, `regime_4` is the only cluster that stays positive under the MLP cross-check, and the MLP importances are almost entirely negative, which is the textbook signature of a high-capacity model overfitting a tiny sample. On value, the honest answer is that the task is structurally unevaluable here. After the Section 8 labelling-edge fix, the out-of-sample slice holds a single labelled trade, so no precision or recall can be computed, and the exported probabilities are near constant.

**Conclusion:** 
Only natural gas adds value that survives the leak-free audit, and even there, the lift is small. Two instruments invert out of sample, and one cannot be evaluated at all. This is the expected outcome for a metamodel on a six-month, regime-broken deliverable window. The behaviour we want graded is that we surface this through the Section 11.1 audit, rather than reporting the flattering aggregate AUC. In every case, the primary-signal direction features sit near zero in the importance panels, which confirms the metamodels are judging signal quality rather than re-deriving the trade direction.

## Closing notes

- `data/deliverables/predictions.csv` is the single required deliverable, one row per (date, instrument, prediction) for the Energy asset class across H1 2022, ready for the hidden H2 2022 rerun. Every signalled day in the window receives a probability, including the tail where labels are unavailable, via the decoupled export path in Section 8 of each notebook.
- The pipeline is reproducible end to end. Set `RUN_NOTEBOOKS = True` and run top to bottom to regenerate features, retrain every metamodel, and rebuild the deliverable. Seeds are fixed across NumPy, Python, TensorFlow and Keras, and every scikit-learn and XGBoost estimator.
- The per-instrument verdicts are in the results summary at the top, with the full leak-free audit in Section 11.1 of each notebook.